# Base Tracer Interfaces

The `base.py` module defines synchronous and asynchronous base classes for tracing LangChain runs.

The tracer classes create and update `Run` objects for chat models, language models, chains, tools, and retrievers. Root runs are persisted when they finish, while child runs are maintained within the tracer's run map and connected to their parent runs.

## Re-exports

1. `TracerException`: Re-exports the tracing exception class from `langchain_core.exceptions`.

# BaseTracer

`BaseTracer` is the synchronous base interface for LangChain tracers.

It implements callback-handler methods that construct, update, complete, and return `Run` objects. Subclasses must implement `_persist_run` to store completed root runs.

## Bases

- `_TracerCore`
- `BaseCallbackHandler`
- `ABC`

### Methods

1. `_persist_run`: Persists a completed root run.

   Subclasses must implement this protected abstract method.

   * **Syntax:**
     ```python
     @abstractmethod
     _persist_run(
         self,
         run: Run # Completed root run to persist
     ) -> None
     ```

2. `on_chat_model_start`: Creates and starts a trace for a chat-model run.

   The run is added to the trace, the chat-model start hook is called, and the created `Run` is returned. Chat-model completion is handled through `on_llm_end` rather than a separate chat-model end callback.

   * **Syntax:**
     ```python
     on_chat_model_start(
         self,
         serialized: dict[str, Any], # Serialized chat model
         messages: list[
             list[BaseMessage]
         ], # Message batches supplied to the model
         *,
         run_id: UUID, # Unique run identifier
         tags: list[str] | None = None, # Optional run tags
         parent_run_id: UUID | None = None, # Optional parent-run identifier
         metadata: dict[str, Any] | None = None, # Optional run metadata
         name: str | None = None, # Optional run name
         **kwargs: Any # Additional run properties
     ) -> Run
     ```

3. `on_llm_start`: Creates and starts a trace for a text-LLM run.

   The LLM start hook is called after the run is registered.

   * **Syntax:**
     ```python
     on_llm_start(
         self,
         serialized: dict[str, Any], # Serialized language model
         prompts: list[str], # Prompts supplied to the model
         *,
         run_id: UUID, # Unique run identifier
         tags: list[str] | None = None, # Optional run tags
         parent_run_id: UUID | None = None, # Optional parent-run identifier
         metadata: dict[str, Any] | None = None, # Optional run metadata
         name: str | None = None, # Optional run name
         **kwargs: Any # Additional run properties
     ) -> Run
     ```

4. `on_llm_new_token`: Adds a streaming token event to an LLM or chat-model run.

   The token may be plain text or structured content blocks. The updated run is passed to the token hook and returned.

   * **Syntax:**
     ```python
     on_llm_new_token(
         self,
         token: str
         | list[
             str
             | dict[str, Any]
         ], # Token or structured content blocks
         *,
         chunk: GenerationChunk
         | ChatGenerationChunk
         | None = None, # Optional generation chunk
         run_id: UUID, # Run receiving the token event
         parent_run_id: UUID | None = None, # Optional parent-run identifier
         **kwargs: Any # Additional callback arguments
     ) -> Run
     ```

5. `on_retry`: Adds a retry event to an LLM run and returns the updated run.
   * **Syntax:**
     ```python
     on_retry(
         self,
         retry_state: RetryCallState, # Tenacity retry state
         *,
         run_id: UUID, # Run receiving the retry event
         **kwargs: Any # Additional callback arguments
     ) -> Run
     ```

6. `on_llm_end`: Completes an LLM or chat-model run successfully.

   The run is finalized, removed from active trace bookkeeping, persisted when it is a root run, passed to the LLM-end hook, and returned.

   * **Syntax:**
     ```python
     on_llm_end(
         self,
         response: LLMResult, # Completed model response
         *,
         run_id: UUID, # Run to complete
         **kwargs: Any # Additional callback arguments
     ) -> Run
     ```

7. `on_llm_error`: Completes an LLM or chat-model run with an error.

   An optional partial response may be supplied through `kwargs["response"]`. The errored run is finalized, passed to the error hook, and returned.

   * **Syntax:**
     ```python
     on_llm_error(
         self,
         error: BaseException, # Error raised during model execution
         *,
         run_id: UUID, # Run to complete
         **kwargs: Any # Additional callback arguments
     ) -> Run
     ```

8. `on_chain_start`: Creates and starts a trace for a chain run.

   The optional `run_type` can identify a specialized chain-like operation.

   * **Syntax:**
     ```python
     on_chain_start(
         self,
         serialized: dict[str, Any], # Serialized chain
         inputs: dict[str, Any], # Chain inputs
         *,
         run_id: UUID, # Unique run identifier
         tags: list[str] | None = None, # Optional run tags
         parent_run_id: UUID | None = None, # Optional parent-run identifier
         metadata: dict[str, Any] | None = None, # Optional run metadata
         run_type: str | None = None, # Optional specialized run type
         name: str | None = None, # Optional run name
         **kwargs: Any # Additional run properties
     ) -> Run
     ```

9. `on_chain_end`: Completes a chain run successfully and returns the finalized run.

   Optional `inputs` may be supplied when they were not available at the start callback.

   * **Syntax:**
     ```python
     on_chain_end(
         self,
         outputs: dict[str, Any], # Chain outputs
         *,
         run_id: UUID, # Run to complete
         inputs: dict[str, Any] | None = None, # Optional chain inputs
         **kwargs: Any # Additional callback arguments
     ) -> Run
     ```

10. `on_chain_error`: Completes a chain run with an error and returns the finalized run.
    * **Syntax:**
      ```python
      on_chain_error(
          self,
          error: BaseException, # Error raised during chain execution
          *,
          inputs: dict[str, Any] | None = None, # Optional chain inputs
          run_id: UUID, # Run to complete
          **kwargs: Any # Additional callback arguments
      ) -> Run
      ```

11. `on_tool_start`: Creates and starts a trace for a tool run.

    The tool input may be supplied as a string and optionally as a structured input dictionary.

    * **Syntax:**
      ```python
      on_tool_start(
          self,
          serialized: dict[str, Any], # Serialized tool
          input_str: str, # String representation of the tool input
          *,
          run_id: UUID, # Unique run identifier
          tags: list[str] | None = None, # Optional run tags
          parent_run_id: UUID | None = None, # Optional parent-run identifier
          metadata: dict[str, Any] | None = None, # Optional run metadata
          name: str | None = None, # Optional run name
          inputs: dict[str, Any] | None = None, # Optional structured tool input
          **kwargs: Any # Additional run properties
      ) -> Run
      ```

12. `on_tool_end`: Completes a tool run successfully and returns the finalized run.
    * **Syntax:**
      ```python
      on_tool_end(
          self,
          output: Any, # Tool output
          *,
          run_id: UUID, # Run to complete
          **kwargs: Any # Additional callback arguments
      ) -> Run
      ```

13. `on_tool_error`: Completes a tool run with an error and returns the finalized run.
    * **Syntax:**
      ```python
      on_tool_error(
          self,
          error: BaseException, # Error raised during tool execution
          *,
          run_id: UUID, # Run to complete
          **kwargs: Any # Additional callback arguments
      ) -> Run
      ```

14. `on_retriever_start`: Creates and starts a trace for a retriever run.
    * **Syntax:**
      ```python
      on_retriever_start(
          self,
          serialized: dict[str, Any], # Serialized retriever
          query: str, # Retrieval query
          *,
          run_id: UUID, # Unique run identifier
          parent_run_id: UUID | None = None, # Optional parent-run identifier
          tags: list[str] | None = None, # Optional run tags
          metadata: dict[str, Any] | None = None, # Optional run metadata
          name: str | None = None, # Optional run name
          **kwargs: Any # Additional run properties
      ) -> Run
      ```

15. `on_retriever_error`: Completes a retriever run with an error and returns the finalized run.
    * **Syntax:**
      ```python
      on_retriever_error(
          self,
          error: BaseException, # Error raised during retrieval
          *,
          run_id: UUID, # Run to complete
          **kwargs: Any # Additional callback arguments
      ) -> Run
      ```

16. `on_retriever_end`: Completes a retriever run successfully and returns the finalized run.
    * **Syntax:**
      ```python
      on_retriever_end(
          self,
          documents: Sequence[Document], # Retrieved documents
          *,
          run_id: UUID, # Run to complete
          **kwargs: Any # Additional callback arguments
      ) -> Run
      ```

17. `__deepcopy__`: Returns the same tracer instance rather than creating a deep copy.
    * **Syntax:**
      ```python
      __deepcopy__(
          self,
          memo: dict[int, Any] | None = None # Optional copy memo
      ) -> BaseTracer
      ```

18. `__copy__`: Returns the same tracer instance rather than creating a shallow copy.
    * **Syntax:**
      ```python
      __copy__(
          self
      ) -> BaseTracer
      ```

# AsyncBaseTracer

`AsyncBaseTracer` is the asynchronous base interface for LangChain tracers.

It creates and updates the same run types as `BaseTracer`, but persistence and lifecycle hooks are asynchronous. Start and completion bookkeeping generally run concurrently with the corresponding type-specific hook through `asyncio.gather`.

Subclasses must implement `_persist_run`. They may also override the protected `_on_*` hooks to process specific lifecycle events.

## Bases

- `_TracerCore`
- `AsyncCallbackHandler`
- `ABC`

### Methods

1. `_persist_run`: Asynchronously persists a completed root run.

   Subclasses must implement this protected abstract method.

   * **Syntax:**
     ```python
     @abstractmethod
     async _persist_run(
         self,
         run: Run # Completed root run to persist
     ) -> None
     ```

2. `on_chat_model_start`: Creates and asynchronously starts a chat-model trace.

   Trace registration and the chat-model start hook run concurrently. The created `Run` is returned.

   * **Syntax:**
     ```python
     async on_chat_model_start(
         self,
         serialized: dict[str, Any], # Serialized chat model
         messages: list[
             list[BaseMessage]
         ], # Message batches supplied to the model
         *,
         run_id: UUID, # Unique run identifier
         parent_run_id: UUID | None = None, # Optional parent-run identifier
         tags: list[str] | None = None, # Optional run tags
         metadata: dict[str, Any] | None = None, # Optional run metadata
         name: str | None = None, # Optional run name
         **kwargs: Any # Additional run properties
     ) -> Any
     ```

3. `on_llm_start`: Creates and asynchronously starts a text-LLM trace.

   Trace registration and the LLM start hook run concurrently.

   * **Syntax:**
     ```python
     async on_llm_start(
         self,
         serialized: dict[str, Any], # Serialized language model
         prompts: list[str], # Prompts supplied to the model
         *,
         run_id: UUID, # Unique run identifier
         parent_run_id: UUID | None = None, # Optional parent-run identifier
         tags: list[str] | None = None, # Optional run tags
         metadata: dict[str, Any] | None = None, # Optional run metadata
         **kwargs: Any # Additional run properties
     ) -> None
     ```

4. `on_llm_new_token`: Adds a streaming token event and awaits the asynchronous token hook.
   * **Syntax:**
     ```python
     async on_llm_new_token(
         self,
         token: str
         | list[
             str
             | dict[str, Any]
         ], # Token or structured content blocks
         *,
         chunk: GenerationChunk
         | ChatGenerationChunk
         | None = None, # Optional generation chunk
         run_id: UUID, # Run receiving the token event
         parent_run_id: UUID | None = None, # Optional parent-run identifier
         **kwargs: Any # Additional callback arguments
     ) -> None
     ```

5. `on_retry`: Adds a retry event to an LLM run.
   * **Syntax:**
     ```python
     async on_retry(
         self,
         retry_state: RetryCallState, # Tenacity retry state
         *,
         run_id: UUID, # Run receiving the retry event
         **kwargs: Any # Additional callback arguments
     ) -> None
     ```

6. `on_llm_end`: Completes an LLM or chat-model run successfully.

   Final trace bookkeeping and `_on_llm_end` run concurrently.

   * **Syntax:**
     ```python
     async on_llm_end(
         self,
         response: LLMResult, # Completed model response
         *,
         run_id: UUID, # Run to complete
         parent_run_id: UUID | None = None, # Optional parent-run identifier
         tags: list[str] | None = None, # Optional run tags
         **kwargs: Any # Additional callback arguments
     ) -> None
     ```

7. `on_llm_error`: Completes an LLM or chat-model run with an error.

   Final trace bookkeeping and `_on_llm_error` run concurrently.

   * **Syntax:**
     ```python
     async on_llm_error(
         self,
         error: BaseException, # Error raised during model execution
         *,
         run_id: UUID, # Run to complete
         parent_run_id: UUID | None = None, # Optional parent-run identifier
         tags: list[str] | None = None, # Optional run tags
         **kwargs: Any # Additional callback arguments
     ) -> None
     ```

8. `on_chain_start`: Creates and asynchronously starts a chain trace.

   Trace registration and `_on_chain_start` run concurrently.

   * **Syntax:**
     ```python
     async on_chain_start(
         self,
         serialized: dict[str, Any], # Serialized chain
         inputs: dict[str, Any], # Chain inputs
         *,
         run_id: UUID, # Unique run identifier
         tags: list[str] | None = None, # Optional run tags
         parent_run_id: UUID | None = None, # Optional parent-run identifier
         metadata: dict[str, Any] | None = None, # Optional run metadata
         run_type: str | None = None, # Optional specialized run type
         name: str | None = None, # Optional run name
         **kwargs: Any # Additional run properties
     ) -> None
     ```

9. `on_chain_end`: Completes a chain run successfully.

   Final trace bookkeeping and `_on_chain_end` run concurrently.

   * **Syntax:**
     ```python
     async on_chain_end(
         self,
         outputs: dict[str, Any], # Chain outputs
         *,
         run_id: UUID, # Run to complete
         inputs: dict[str, Any] | None = None, # Optional chain inputs
         **kwargs: Any # Additional callback arguments
     ) -> None
     ```

10. `on_chain_error`: Completes a chain run with an error.

    Final trace bookkeeping and `_on_chain_error` run concurrently.

    * **Syntax:**
      ```python
      async on_chain_error(
          self,
          error: BaseException, # Error raised during chain execution
          *,
          inputs: dict[str, Any] | None = None, # Optional chain inputs
          run_id: UUID, # Run to complete
          **kwargs: Any # Additional callback arguments
      ) -> None
      ```

11. `on_tool_start`: Creates and asynchronously starts a tool trace.

    Trace registration and `_on_tool_start` run concurrently.

    * **Syntax:**
      ```python
      async on_tool_start(
          self,
          serialized: dict[str, Any], # Serialized tool
          input_str: str, # String representation of the tool input
          *,
          run_id: UUID, # Unique run identifier
          tags: list[str] | None = None, # Optional run tags
          parent_run_id: UUID | None = None, # Optional parent-run identifier
          metadata: dict[str, Any] | None = None, # Optional run metadata
          name: str | None = None, # Optional run name
          inputs: dict[str, Any] | None = None, # Optional structured tool input
          **kwargs: Any # Additional run properties
      ) -> None
      ```

12. `on_tool_end`: Completes a tool run successfully.

    Final trace bookkeeping and `_on_tool_end` run concurrently.

    * **Syntax:**
      ```python
      async on_tool_end(
          self,
          output: Any, # Tool output
          *,
          run_id: UUID, # Run to complete
          **kwargs: Any # Additional callback arguments
      ) -> None
      ```

13. `on_tool_error`: Completes a tool run with an error.

    Final trace bookkeeping and `_on_tool_error` run concurrently.

    * **Syntax:**
      ```python
      async on_tool_error(
          self,
          error: BaseException, # Error raised during tool execution
          *,
          run_id: UUID, # Run to complete
          parent_run_id: UUID | None = None, # Optional parent-run identifier
          tags: list[str] | None = None, # Optional run tags
          **kwargs: Any # Additional callback arguments
      ) -> None
      ```

14. `on_retriever_start`: Creates and asynchronously starts a retriever trace.

    Trace registration and `_on_retriever_start` run concurrently.

    * **Syntax:**
      ```python
      async on_retriever_start(
          self,
          serialized: dict[str, Any], # Serialized retriever
          query: str, # Retrieval query
          *,
          run_id: UUID, # Unique run identifier
          parent_run_id: UUID | None = None, # Optional parent-run identifier
          tags: list[str] | None = None, # Optional run tags
          metadata: dict[str, Any] | None = None, # Optional run metadata
          name: str | None = None, # Optional run name
          **kwargs: Any # Additional run properties
      ) -> None
      ```

15. `on_retriever_error`: Completes a retriever run with an error.

    Final trace bookkeeping and `_on_retriever_error` run concurrently.

    * **Syntax:**
      ```python
      async on_retriever_error(
          self,
          error: BaseException, # Error raised during retrieval
          *,
          run_id: UUID, # Run to complete
          parent_run_id: UUID | None = None, # Optional parent-run identifier
          tags: list[str] | None = None, # Optional run tags
          **kwargs: Any # Additional callback arguments
      ) -> None
      ```

16. `on_retriever_end`: Completes a retriever run successfully.

    Final trace bookkeeping and `_on_retriever_end` run concurrently.

    * **Syntax:**
      ```python
      async on_retriever_end(
          self,
          documents: Sequence[Document], # Retrieved documents
          *,
          run_id: UUID, # Run to complete
          parent_run_id: UUID | None = None, # Optional parent-run identifier
          tags: list[str] | None = None, # Optional run tags
          **kwargs: Any # Additional callback arguments
      ) -> None
      ```

### Protected Lifecycle Hooks

The following asynchronous hooks have no default processing behaviour. Subclasses may override them to react to particular lifecycle events.

1. `_on_run_create`: Processes a newly created run.
   * **Syntax:**
     ```python
     async _on_run_create(
         self,
         run: Run # Newly created run
     ) -> None
     ```

2. `_on_run_update`: Processes a completed or otherwise updated run.
   * **Syntax:**
     ```python
     async _on_run_update(
         self,
         run: Run # Updated run
     ) -> None
     ```

3. `_on_llm_start`: Processes an LLM run when it starts.
   * **Syntax:**
     ```python
     async _on_llm_start(
         self,
         run: Run # Started LLM run
     ) -> None
     ```

4. `_on_llm_end`: Processes an LLM or chat-model run after successful completion.
   * **Syntax:**
     ```python
     async _on_llm_end(
         self,
         run: Run # Completed model run
     ) -> None
     ```

5. `_on_llm_error`: Processes an LLM or chat-model run after an error.
   * **Syntax:**
     ```python
     async _on_llm_error(
         self,
         run: Run # Errored model run
     ) -> None
     ```

6. `_on_llm_new_token`: Processes a newly generated LLM or chat-model token.
   * **Syntax:**
     ```python
     async _on_llm_new_token(
         self,
         run: Run, # Run receiving the token
         token: str
         | list[
             str
             | dict[str, Any]
         ], # Token or structured content blocks
         chunk: GenerationChunk
         | ChatGenerationChunk
         | None # Optional generation chunk
     ) -> None
     ```

7. `_on_chain_start`: Processes a chain run when it starts.
   * **Syntax:**
     ```python
     async _on_chain_start(
         self,
         run: Run # Started chain run
     ) -> None
     ```

8. `_on_chain_end`: Processes a successfully completed chain run.
   * **Syntax:**
     ```python
     async _on_chain_end(
         self,
         run: Run # Completed chain run
     ) -> None
     ```

9. `_on_chain_error`: Processes a chain run after an error.
   * **Syntax:**
     ```python
     async _on_chain_error(
         self,
         run: Run # Errored chain run
     ) -> None
     ```

10. `_on_tool_start`: Processes a tool run when it starts.
    * **Syntax:**
      ```python
      async _on_tool_start(
          self,
          run: Run # Started tool run
      ) -> None
      ```

11. `_on_tool_end`: Processes a successfully completed tool run.
    * **Syntax:**
      ```python
      async _on_tool_end(
          self,
          run: Run # Completed tool run
      ) -> None
      ```

12. `_on_tool_error`: Processes a tool run after an error.
    * **Syntax:**
      ```python
      async _on_tool_error(
          self,
          run: Run # Errored tool run
      ) -> None
      ```

13. `_on_chat_model_start`: Processes a chat-model run when it starts.
    * **Syntax:**
      ```python
      async _on_chat_model_start(
          self,
          run: Run # Started chat-model run
      ) -> None
      ```

14. `_on_retriever_start`: Processes a retriever run when it starts.
    * **Syntax:**
      ```python
      async _on_retriever_start(
          self,
          run: Run # Started retriever run
      ) -> None
      ```

15. `_on_retriever_end`: Processes a successfully completed retriever run.
    * **Syntax:**
      ```python
      async _on_retriever_end(
          self,
          run: Run # Completed retriever run
      ) -> None
      ```

16. `_on_retriever_error`: Processes a retriever run after an error.
    * **Syntax:**
      ```python
      async _on_retriever_error(
          self,
          run: Run # Errored retriever run
      ) -> None
      ```